# Clasificador de dígitos MNIST con imágenes propias

**Objetivo:** reutilizar el modelo denso de dos capas (`Dense(64)` -> `Dense(16)` -> `Dense(10)`) y validarlo con imágenes que **no** vienen del dataset: fotos, capturas de pantalla o dibujos hechos a mano, con una interfaz para cargarlas.

## Qué cambia respecto del notebook original

| # | Cambio | Por qué |
|---|---|---|
| 1 | Normalización `/255` en train y test | El original entrenaba con píxeles en `[0,255]`. Por eso la accuracy arrancaba en 41% y saltaba de golpe en la época 5. |
| 2 | `validation_split` + `EarlyStopping` | 10 épocas fijas no son una decisión, son una apuesta. |
| 3 | Evaluación real sobre `x_test` | El 96.86% del original era accuracy de **train**. No medía generalización. |
| 4 | Función de preprocesamiento | Convierte una imagen cualquiera al formato exacto de MNIST. |
| 5 | Interfaz Gradio | Cargar la imagen, ver la predicción y ver qué ve el modelo. |

## Cómo ejecutar en Colab

`Entorno de ejecución > Ejecutar todas`. La interfaz aparece al final.
Si ya entrenaste en esta sesión, **no re-entrenes**: saltá a la celda de carga del modelo.

## 1. Datos y normalización

MNIST entrega las imágenes como `uint8` en `[0, 255]`. La red no necesita ese rango: lo que necesita es que **train e inferencia usen exactamente la misma escala**. Normalizar acá y no normalizar en la interfaz es el error más común de este ejercicio.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalizar a [0, 1] y aplanar 28x28 -> 784
x_train = (x_train.astype('float32') / 255.0).reshape(-1, 28 * 28)
x_test = (x_test.astype('float32') / 255.0).reshape(-1, 28 * 28)

print('x_train:', x_train.shape, '| rango:', float(x_train.min()), '-', float(x_train.max()))
print('x_test :', x_test.shape)
print('clases :', np.unique(y_train))

: 

## 2. Modelo: la misma arquitectura de dos capas

Se mantiene intacta: `784 -> 64 (relu) -> 16 (relu) -> 10 (softmax)`, 51,450 parámetros.
Solo se le ponen nombres a las capas para que el `summary` sea legible y el modelo sea identificable al guardarlo.

In [ ]:
In = tf.keras.layers.Input(shape=(784,), name='entrada')
Cp1 = tf.keras.layers.Dense(64, activation='relu', name='oculta_1')(In)
Cp2 = tf.keras.layers.Dense(16, activation='relu', name='oculta_2')(Cp1)
Out = tf.keras.layers.Dense(10, activation='softmax', name='salida')(Cp2)

Model = tf.keras.Model(inputs=In, outputs=Out, name='mlp_dos_capas')
Model.summary()

Model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

## 3. Entrenamiento

`EarlyStopping` corta cuando la validación deja de mejorar y **restaura los mejores pesos**. Eso evita quedarse con una época sobreentrenada, que es exactamente lo que le pasaba al notebook original.

In [ ]:
# --- Aumento de datos OPCIONAL ---
# Activá esto si tus fotos tienen rotaciones o trazos bastante distintos a MNIST.
# No cambia la arquitectura: solo agrega copias transformadas al set de entrenamiento.
USE_AUGMENTATION = False

x_tr, y_tr = x_train, y_train

if USE_AUGMENTATION:
    aug = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.RandomRotation(0.05, fill_mode='constant', fill_value=0.0),
        tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode='constant', fill_value=0.0),
    ], name='augment')

    extra = aug(x_train.reshape(-1, 28, 28, 1), training=True)
    extra = extra.numpy().reshape(-1, 28 * 28)

    x_tr = np.concatenate([x_train, extra])
    y_tr = np.concatenate([y_train, y_train])
    print('Con aumento:', x_tr.shape)
else:
    print('Sin aumento:', x_tr.shape)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = Model.fit(
    x_tr, y_tr,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validación')
axes[0].set_title('Loss')
axes[0].set_xlabel('época')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='validación')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('época')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Evaluación sobre el set de test

Esta es la métrica que vale. La matriz de confusión muestra **qué dígitos confunde** el modelo, que es información mucho más útil que un único número.

In [ ]:
loss, acc = Model.evaluate(x_test, y_test, verbose=0)
print(f'Accuracy en TEST: {acc:.4f}  |  loss: {loss:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = np.argmax(Model.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d'
)
ax.set_title(f'Matriz de confusión — accuracy test: {acc:.2%}')
plt.show()

## 5. Guardar y cargar el modelo

Colab borra el entorno cuando se cierra la sesión. Guardar el `.keras` permite volver a la interfaz sin entrenar de nuevo.

In [ ]:
MODEL_PATH = 'mlp_dos_capas.keras'

Model.save(MODEL_PATH)
print('Modelo guardado en:', MODEL_PATH)

# En una sesión nueva, descomentá esta línea para no re-entrenar:
# Model = tf.keras.models.load_model(MODEL_PATH)
# print('Modelo cargado. Params:', Model.count_params())

## 6. Preprocesamiento: el paso que decide todo

El modelo no ve "un dibujo del número 7". Ve 784 números. MNIST tiene reglas muy específicas y la red las aprendió:

| Característica | MNIST | Tu foto |
|---|---|---|
| Tamaño | 28x28 | cualquiera |
| Color | escala de grises | color |
| Fondo | negro (0) | casi siempre blanco |
| Trazo | blanco (255) | oscuro |
| Escala | dígito ocupa ~20x20 | variable |
| Posición | centrado por centro de masa | donde caiga |

Si no replicás esas seis condiciones, el modelo recibe una distribución distinta a la que aprendió y predice mal **aunque tenga 98% en test**. Esto se llama *data drift* y es la causa número uno de que "funcione en el notebook y no en la vida real".

La función de abajo hace las seis conversiones.

In [ ]:
import cv2
from PIL import Image


def _translate(img, dy, dx):
    """Desplaza la imagen sin envolver: los bordes que salen quedan en cero."""
    out = np.zeros_like(img)
    h, w = img.shape
    out[max(0, dy):h + min(0, dy), max(0, dx):w + min(0, dx)] = \
        img[max(0, -dy):h + min(0, -dy), max(0, -dx):w + min(0, -dx)]
    return out


def preprocess_image(img):
    """Convierte una imagen arbitraria al formato exacto de MNIST.

    Devuelve:
        x        -> ndarray float32 (1, 784) en [0, 1], listo para Model.predict
        preview  -> ndarray float32 (28, 28) en [0, 1], para inspección visual
    """
    # 1. Escala de grises
    if isinstance(img, Image.Image):
        gray = np.array(img.convert('L'))
    else:
        arr = np.asarray(img)
        if arr.ndim == 3:
            if arr.shape[-1] == 3:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            elif arr.shape[-1] == 4:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2GRAY)
            else:
                arr = arr[..., 0]
        gray = arr.astype(np.uint8)

    if gray.size == 0:
        raise ValueError('La imagen está vacía.')

    # 2. Binarizar con Otsu: separa trazo de fondo sin umbral mágico
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 3. Forzar polaridad MNIST: trazo blanco, fondo negro
    if binary.mean() > 127:
        binary = cv2.bitwise_not(binary)

    # 4. Recortar al bounding box del trazo
    coords = cv2.findNonZero(binary)
    if coords is None:
        raise ValueError('No se detectó ningún trazo. Usá más contraste o un fondo liso.')
    x0, y0, w, h = cv2.boundingRect(coords)
    if w < 3 or h < 3:
        raise ValueError('El trazo detectado es demasiado pequeño. Acercá la imagen.')
    digit = binary[y0:y0 + h, x0:x0 + w]

    # 5. Escalar a 20 px el lado mayor, manteniendo proporción (regla MNIST)
    scale = 20.0 / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 6. Pegar centrado sobre un lienzo 28x28
    canvas = np.zeros((28, 28), dtype=np.float32)
    top = (28 - new_h) // 2
    left = (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = digit.astype(np.float32)

    # 7. Centrar por centro de masa (igual que hizo el dataset original)
    total = float(canvas.sum())
    if total > 0:
        ys, xs = np.indices(canvas.shape)
        cy = float((canvas * ys).sum()) / total
        cx = float((canvas * xs).sum()) / total
        canvas = _translate(canvas, int(round(13.5 - cy)), int(round(13.5 - cx)))

    # 8. Misma normalización que en el entrenamiento
    canvas = (canvas / 255.0).astype(np.float32)
    return canvas.reshape(1, 784), canvas


print('preprocess_image lista.')

### Verificación del preprocesamiento

Antes de confiar en la interfaz, comprobá que la función **reconstruye correctamente** una imagen que ya sabe que funciona. Si acá falla, el problema no es el modelo.

In [ ]:
idx = 7
raw = (x_test[idx].reshape(28, 28) * 255).astype(np.uint8)

x_in, preview = preprocess_image(raw)
pred = int(np.argmax(Model.predict(x_in, verbose=0)[0]))
real = int(y_test[idx])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(raw, cmap='gray'); axes[0].set_title('Original (MNIST)'); axes[0].axis('off')
axes[1].imshow(preview, cmap='gray'); axes[1].set_title('Preprocesada 28x28'); axes[1].axis('off')
plt.suptitle(f'Predicción: {pred}  |  real: {real}', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Interfaz para cargar imágenes propias

Gradio levanta una web dentro de Colab con carga por archivo, portapapeles y cámara. La salida muestra:

1. Las 3 probabilidades más altas.
2. **La imagen de 28x28 que realmente ve el modelo** — esto es clave para depurar.
3. Un aviso cuando la confianza es baja.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr


def classify(img):
    if img is None:
        raise gr.Error('Cargá una imagen primero.')

    try:
        x_in, preview = preprocess_image(img)
    except Exception as exc:
        raise gr.Error(f'No pude procesar la imagen: {exc}')

    probs = Model.predict(x_in, verbose=0)[0]
    top3 = np.argsort(probs)[::-1][:3]
    best = int(top3[0])

    scores = {str(int(i)): float(probs[i]) for i in top3}
    resumen = f'Dígito: {best}   |   confianza: {probs[best]:.2%}'

    if probs[best] < 0.60:
        resumen += '\nConfianza baja: mirá la imagen de la derecha. El preprocesamiento no está logrando el formato MNIST.'

    return scores, preview, resumen


demo = gr.Interface(
    fn=classify,
    inputs=gr.Image(
        type='pil',
        image_mode='L',
        sources=['upload', 'clipboard', 'webcam'],
        label='Imagen con UN solo dígito',
    ),
    outputs=[
        gr.Label(num_top_classes=3, label='Probabilidades'),
        gr.Image(label='Lo que ve el modelo (28x28)', height=260),
        gr.Textbox(label='Resumen', lines=2),
    ],
    title='Clasificador de dígitos — MLP de 2 capas',
    description='Subí una foto, un recorte o un dibujo. Mejor resultado: trazo oscuro sobre fondo blanco liso, un dígito por imagen.',
)

demo.launch(share=True, debug=True)

### Alternativa si Gradio no levanta

Colab a veces bloquea el túnel de Gradio. Este bloque hace lo mismo con el uploader nativo y no depende de ninguna web.

In [ ]:
from google.colab import files
from PIL import Image as PILImage

uploaded = files.upload()

if uploaded:
    name = next(iter(uploaded))
    img = PILImage.open(name)
    x_in, preview = preprocess_image(img)
    probs = Model.predict(x_in, verbose=0)[0]
    pred = int(np.argmax(probs))

    fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
    axes[0].imshow(img.convert('L'), cmap='gray'); axes[0].set_title(name); axes[0].axis('off')
    axes[1].imshow(preview, cmap='gray'); axes[1].axis('off')
    axes[1].set_title(f'Predicción: {pred} ({probs[pred]:.1%})')
    plt.tight_layout()
    plt.show()

## 8. Qué esperar y qué hacer cuando falla

**Un MLP entrenado con MNIST generaliza bien a capturas de MNIST y regular a fotos reales.** No es un bug: es la diferencia entre el dataset y el mundo.

| Síntoma | Causa más probable | Solución |
|---|---|---|
| Siempre predice el mismo dígito | La imagen llega invertida o sin recortar | Mirá el panel "lo que ve el modelo". Si está mayormente blanco, falló la polaridad |
| Confianza baja en todos los casos | Trazo con grosor muy distinto a MNIST | Probar `USE_AUGMENTATION = True` y re-entrenar |
| Falla con fotos pero anda con capturas | Iluminación despareja | Reemplazar Otsu por umbral adaptativo (`cv2.adaptiveThreshold`) |
| Confunde 4/9, 3/5, 7/1 | Ambigüedad real del dataset | Es el techo del modelo, no del código |
| Anda bien y después de reiniciar no | Colab borró el entorno | Cargar `mlp_dos_capas.keras` con `load_model` |

**Mejora siguiente, cuando esto ya funcione:** reemplazar el MLP por una CNN (`Conv2D` -> `MaxPooling2D` -> `Flatten` -> `Dense`). El dense aplana la imagen y pierde la noción de píxeles vecinos; una CNN llega a 99%+ y tolera mucho mejor fotos reales.